In [1]:
from google.colab import userdata
openai_key = userdata.get('OPENAI_API_KEY')
google_key = userdata.get('GOOGLE_API_KEY')
pinecone_api_key = userdata.get('PINECONE_API_KEY')

In [2]:
!pip -q install pinecone sentence-transformers langchain langchain-google-genai pyyaml

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 742.7/742.7 kB 33.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 280.9/280.9 kB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 4.3 MB/s eta 0:00:00


In [3]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [5]:
#Define your data path
data_dir = "/content/drive/MyDrive/Session 3 Rag/data"

### Step 1: Access your Vector DB remotely

In [6]:
from pinecone import Pinecone, ServerlessSpec
import time

pc = Pinecone(api_key=pinecone_api_key)
# I want to meet vector-hybrid
index_name = 'vector-docs'

### Step 2: Check if Index exists else create it

In [7]:
# Step 2. Check if index exists, if not, create it
if index_name not in [idx.name for idx in pc.list_indexes()]:
    print(f"Index '{index_name}' not found. Creating it...")
    pc.create_index(
        name=index_name,
        dimension=384, # Must match paraphrase-MiniLM-L6-v2
        metric='cosine',
        spec=ServerlessSpec(cloud='aws', region='us-east-1')
    )
    # Optional
    # Wait for index to spin up
    while not pc.describe_index(index_name).status['ready']:
        time.sleep(10)
    print("Index created and ready.")
else:
    print(f"Index '{index_name}' already exists.")

# Connect to the index
index = pc.Index(index_name)

Index 'vector-docs' not found. Creating it...
Index created and ready.


### Step 3: Organize you DOCS

In [8]:
!pip install langchain-text-splitters

In [ ]:
# import os
# def simple_chunker(text, size = 500):
#   return [text[i:i+size] for i in range(0, len(text), size)]
# ## All the files
# docs = []
# # filename = os.listdir(data_dir)[0]
# for filename in os.listdir(data_dir):
#   print(filename)
#   if filename.endswith(".txt"):
#     # Replace this reading part with a function to read from html/word/pdf
#     with open(os.path.join(data_dir, filename), "r", encoding="utf-8") as f:
#       content = f.read()
#       chunks = simple_chunker(content)
#       for i, chunk in enumerate(chunks):
#         docs.append({"id": f"{filename}_chunk_{i}",
#                     "text": chunk
#       })

loan_against_property.txt
FAQs.txt
home_loan.txt
personal_loan.txt
eligibility_documents.txt
two_wheeler_loan.txt
no_cost_emi.txt
about_us.txt


### NEW ways of chunking - Textsplitters from LangChain

In [9]:
import os
from langchain_text_splitters import RecursiveCharacterTextSplitter


def build_langchain_splitter(chunk_size=500, chunk_overlap=80):
    """
    Builds a generic text splitter using LangChain's recommended
    RecursiveCharacterTextSplitter.

    chunk_size and chunk_overlap here are character-based.
    """
    return RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        # separators = ["\n\n", "\n", " ", ""],
        length_function=len,   # measure by characters
        is_separator_regex=False
    )


def chunk_text_langchain(text, splitter):
    """
    Splits one text string into chunks.
    Returns a list of strings.
    """
    if not text or not text.strip():
        return []
    return splitter.split_text(text)

def prepare_docs_langchain(data_dir, chunk_size=500, chunk_overlap=200):
    """
    Reads all .txt files from data_dir, chunks them using LangChain,
    and returns output in the SAME shape as your current Step 3:

    [
        {"id": "file1.txt_chunk_0", "text": "..."},
        {"id": "file1.txt_chunk_1", "text": "..."},
        ...
    ]
    """
    #This will bring the Objet of Recursive text char spillter which will be used further for spliting
    splitter = build_langchain_splitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )

    docs = []

    for filename in os.listdir(data_dir):
        print(filename)

        if filename.endswith(".txt"):
            file_path = os.path.join(data_dir, filename)

            with open(file_path, "r", encoding="utf-8") as f:
                content = f.read()
                # content = content.replace("\n", "")
            chunks = chunk_text_langchain(content, splitter)

            for i, chunk in enumerate(chunks):
                if chunk.strip():  # skip empty chunks
                    docs.append({
                        "id": f"{filename}_chunk_{i}",
                        "text": chunk
                    })

    return docs

In [10]:
# lst = [0,1,4,11,13,15,91, 12]
# lst[0:5]
# # docs[:5]

In [11]:
docs = prepare_docs_langchain(data_dir, chunk_size=500, chunk_overlap=80)

personal_loan.txt
two_wheeler_loan.txt
no_cost_emi.txt
home_loan.txt
FAQs.txt
eligibility_documents.txt
loan_against_property.txt
about_us.txt


In [12]:
docs[:5]

[{'id': 'personal_loan.txt_chunk_0',
  'text': 'BrightBridge Finance — Personal Loan\nSimple loans. Clear terms. Fast decisions.\n\nOverview\nA Personal Loan is an unsecured loan designed for quick access to funds without the need to pledge collateral. It is suitable when you need flexible financing for planned or unplanned expenses—medical needs, education, travel, wedding expenses, home improvement, or consolidating multiple debts into a single EMI.'},
 {'id': 'personal_loan.txt_chunk_1',
  'text': 'Why customers choose our Personal Loan\n• Fast in-principle decision for many profiles once key details are verified\n• No collateral required (unsecured)\n• Flexible tenors and EMI options\n• Transparent communication via Key Fact Statement (KFS) and repayment schedule\n• Digital-first experience (where available), with offline support when needed'},
 {'id': 'personal_loan.txt_chunk_2',
  'text': 'Typical use cases\nDebt consolidation: Convert multiple high-interest dues into one structu

### Continuation of Step 3 - Another Chunking technique

In [13]:
import os
import re
import numpy as np


def split_into_semantic_units(text):
    """
    First try paragraph-level splitting.
    If the text has no real paragraphs, fall back to sentence-like splitting.
    Returns: list[str]
    """
    text = text.strip()
    if not text:
        return []

    paragraphs = [p.strip() for p in re.split(r"\n\s*\n+", text) if p.strip()]
    if len(paragraphs) >= 2:
        return paragraphs

    sentences = re.split(r"(?<=[.!?])\s+", text)
    sentences = [s.strip() for s in sentences if s.strip()]
    return sentences


def cosine_similarity(a, b):
    """
    Cosine similarity between two 1D numpy vectors.
    """
    a_norm = np.linalg.norm(a)
    b_norm = np.linalg.norm(b)

    if a_norm == 0 or b_norm == 0:
        return 0.0

    return float(np.dot(a, b) / (a_norm * b_norm))


def hard_split_large_text(text, hard_max_chunk_chars=1200):
    """
    Safety fallback if one chunk becomes too large.
    Returns: list[str]
    """
    if len(text) <= hard_max_chunk_chars:
        return [text]

    pieces = []
    start = 0
    while start < len(text):
        end = start + hard_max_chunk_chars
        pieces.append(text[start:end].strip())
        start = end

    return [p for p in pieces if p]


def semantic_chunk_text_embedding(
    text,
    model,
    similarity_threshold=0.65,
    min_chunk_chars=300,
    target_chunk_chars=700,
    hard_max_chunk_chars=1200
):
    """
    Embedding-based semantic chunking.

    Logic:
    - split into paragraph/sentence units
    - embed each unit
    - compare adjacent units using cosine similarity
    - merge if semantically close
    - split if topic seems to shift
    - force split if chunk gets too large

    Returns: list[str]
    """
    units = split_into_semantic_units(text)
    if not units:
        return []

    if len(units) == 1:
        return hard_split_large_text(units[0], hard_max_chunk_chars)

    # Encode all units once
    unit_embeddings = model.encode(units)

    chunks = []
    current_chunk = units[0]
    current_chunk_emb = unit_embeddings[0]

    for i in range(1, len(units)):
        next_unit = units[i]
        next_emb = unit_embeddings[i]

        sim = cosine_similarity(current_chunk_emb, next_emb)
        proposed_chunk = current_chunk + "\n\n" + next_unit

        # If current chunk is too small, keep growing it first
        if len(current_chunk) < min_chunk_chars and len(proposed_chunk) <= hard_max_chunk_chars:
            current_chunk = proposed_chunk
            current_chunk_emb = model.encode([current_chunk])[0]
            continue

        # If it becomes too large, close current chunk
        if len(proposed_chunk) > hard_max_chunk_chars:
            chunks.extend(hard_split_large_text(current_chunk, hard_max_chunk_chars))
            current_chunk = next_unit
            current_chunk_emb = next_emb
            continue

        # If still below target size, bias toward merging
        if len(current_chunk) < target_chunk_chars:
            current_chunk = proposed_chunk
            current_chunk_emb = model.encode([current_chunk])[0]
            continue

        # Main semantic decision
        if sim >= similarity_threshold:
            current_chunk = proposed_chunk
            current_chunk_emb = model.encode([current_chunk])[0]
        else:
            chunks.extend(hard_split_large_text(current_chunk, hard_max_chunk_chars))
            current_chunk = next_unit
            current_chunk_emb = next_emb

    if current_chunk.strip():
        chunks.extend(hard_split_large_text(current_chunk, hard_max_chunk_chars))

    return [c.strip() for c in chunks if c.strip()]

# Driver functions
def prepare_docs_semantic_embedding(
    data_dir,
    model,
    similarity_threshold=0.65,
    min_chunk_chars=300,
    target_chunk_chars=700,
    hard_max_chunk_chars=1200 # What if the meaning is not finished
):
    """
    Reads .txt files, applies embedding-based semantic chunking,
    and returns docs in the SAME shape as your current Step 3:

    [
        {"id": "file1.txt_chunk_0", "text": "..."},
        {"id": "file1.txt_chunk_1", "text": "..."},
        ...
    ]
    """
    docs = []

    for filename in os.listdir(data_dir):
        print(filename)

        if filename.endswith(".txt"):
            file_path = os.path.join(data_dir, filename)

            with open(file_path, "r", encoding="utf-8") as f:
                content = f.read()

            chunks = semantic_chunk_text_embedding(
                text=content,
                model=model,
                similarity_threshold=similarity_threshold,
                min_chunk_chars=min_chunk_chars,
                target_chunk_chars=target_chunk_chars,
                hard_max_chunk_chars=hard_max_chunk_chars
            )

            for i, chunk in enumerate(chunks):
                docs.append({
                    "id": f"{filename}_chunk_{i}",
                    "text": chunk
                })

    return docs

In [14]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("paraphrase-MiniLM-L6-v2")

docs = prepare_docs_semantic_embedding(
    data_dir=data_dir,
    model=model,
    similarity_threshold=0.65,
    min_chunk_chars=300,
    target_chunk_chars=700,
    hard_max_chunk_chars=1200
)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

personal_loan.txt
two_wheeler_loan.txt
no_cost_emi.txt
home_loan.txt
FAQs.txt
eligibility_documents.txt
loan_against_property.txt
about_us.txt


In [15]:
docs[:5]

[{'id': 'personal_loan.txt_chunk_0',
  'text': 'BrightBridge Finance — Personal Loan\nSimple loans. Clear terms. Fast decisions.\n\nOverview\nA Personal Loan is an unsecured loan designed for quick access to funds without the need to pledge collateral. It is suitable when you need flexible financing for planned or unplanned expenses—medical needs, education, travel, wedding expenses, home improvement, or consolidating multiple debts into a single EMI.\n\nWhy customers choose our Personal Loan\n• Fast in-principle decision for many profiles once key details are verified\n• No collateral required (unsecured)\n• Flexible tenors and EMI options\n• Transparent communication via Key Fact Statement (KFS) and repayment schedule\n• Digital-first experience (where available), with offline support when needed'},
 {'id': 'personal_loan.txt_chunk_1',
  'text': 'Typical use cases\nDebt consolidation: Convert multiple high-interest dues into one structured EMI.\nEmergency expenses: Medical procedures

## Step 4 - Create EMbeddings

In [16]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("paraphrase-MiniLM-L6-v2")  # 384-dim
embeddings = model.encode([d["text"] for d in docs])  # shape (n, 384)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


### Step 5 - Organize Docs & Embeddings together

- Remove Stop words
- For each chunk find the remaining words and count frequency
- Keep top occuring words as keywords for each chunk

In [17]:
# vectors = [
#     {
#         "id": d["id"],
#         "values": emb.tolist(),
#         "metadata": {"text": d["text"]},
#     }
#     for d, emb in zip(docs, embeddings)
# ]

In [18]:
import re
from collections import Counter


STOPWORDS = {
    "the", "is", "are", "was", "were", "a", "an", "and", "or", "of", "to", "in",
    "for", "on", "with", "as", "by", "at", "from", "that", "this", "it", "be",
    "has", "have", "had", "will", "would", "can", "could", "should", "may",
    "might", "not", "but", "if", "then", "than", "so", "such", "into", "their",
    "there", "about", "over", "under", "between", "during", "using", "used"
}


def parse_doc_id(doc_id):
    """
    Example:
    'loan_policy.txt_chunk_4' -> ('loan_policy.txt', 4)
    """
    if "_chunk_" in doc_id:
        source, chunk_no = doc_id.rsplit("_chunk_", 1)
        try:
            chunk_no = int(chunk_no)
        except ValueError:
            chunk_no = -1
        return source, chunk_no
    return doc_id, -1


def normalize_tokens(text):
    """
    Lowercase and keep useful word-like tokens.
    """
    tokens = re.findall(r"[a-zA-Z][a-zA-Z0-9_-]+", text.lower())
    tokens = [t for t in tokens if len(t) >= 3 and t not in STOPWORDS]
    return tokens


def extract_keywords_simple(text, top_k=8):
    """
    Frequency-based keyword extraction.
    Easy to understand and good enough for metadata tags.
    """
    tokens = normalize_tokens(text)
    freq = Counter(tokens)
    return [word for word, _ in freq.most_common(top_k)]


def infer_doc_type(source_name, default_type="text"):
    """
    Simple doc type inference from filename.
    """
    name = source_name.lower()
    if "faq" in name:
        return "faq"
    if "policy" in name:
        return "policy"
    if "guide" in name or "manual" in name:
        return "guide"
    return default_type


def build_metadata_for_doc(doc, top_k_keywords=8, source_url_map=None, default_type="text"):
    """
    Build flat Pinecone metadata for one chunk.
    """
    source, chunk_no = parse_doc_id(doc["id"])
    keywords = extract_keywords_simple(doc["text"], top_k=top_k_keywords)

    metadata = {
        "text": doc["text"],
        "source": source,
        "chunk_no": chunk_no,
        "keywords": keywords,
        "doc_type": infer_doc_type(source, default_type=default_type),
        "char_count": len(doc["text"])
    }

    if source_url_map is not None and source in source_url_map:
        metadata["source_url"] = source_url_map[source]

    return metadata


def build_vectors_with_metadata(
    docs,
    embeddings,
    top_k_keywords=8,
    source_url_map=None,
    default_type="text"
):
    """
    Convert docs + embeddings into Pinecone upsert payload.
    Final output stays in your required shape for Step 5/6 usage.
    """
    vectors = []

    for doc, emb in zip(docs, embeddings):
        metadata = build_metadata_for_doc(
            doc=doc,
            top_k_keywords=top_k_keywords,
            source_url_map=source_url_map,
            default_type=default_type
        )

        vectors.append({
            "id": doc["id"],
            "values": emb.tolist(),
            "metadata": metadata
        })

    return vectors

In [19]:
# Optional: add real links if you have them
source_url_map = {
    "loan_against_property.txt": "https://your-site.com/loan-policy",
    "emi_faq.txt": "https://your-site.com/emi-faq",
}

vectors = build_vectors_with_metadata(
    docs=docs,
    embeddings=embeddings,
    top_k_keywords=8,
    source_url_map=source_url_map,
    default_type="text"
)

In [20]:
vectors[0]

{'id': 'personal_loan.txt_chunk_0',
 'values': [-0.16001853346824646,
  -0.15592190623283386,
  -0.34689682722091675,
  -0.08383795619010925,
  0.061267171055078506,
  -0.08040299266576767,
  -0.08795418590307236,
  0.20821957290172577,
  0.008421776816248894,
  -0.0952305942773819,
  -0.20468081533908844,
  0.04186609759926796,
  -0.09874413907527924,
  -0.18943367898464203,
  0.04073307663202286,
  -0.14697426557540894,
  -0.08027059584856033,
  -0.0618981271982193,
  -0.041096579283475876,
  0.40586555004119873,
  0.04033280536532402,
  -0.06049187108874321,
  -0.07555662095546722,
  0.14614266157150269,
  0.2627585232257843,
  -0.13455960154533386,
  0.2562854290008545,
  -0.11584264039993286,
  0.138252854347229,
  -0.11064153164625168,
  0.14835774898529053,
  0.2427230030298233,
  0.042246926575899124,
  0.045088671147823334,
  0.11010917276144028,
  0.2507984936237335,
  -0.15914760529994965,
  -0.19423866271972656,
  -0.28037944436073303,
  -0.07380439341068268,
  -0.039995729

In [21]:
index.upsert(vectors)

UpsertResponse(upserted_count=43, _response_info={'raw_headers': {'date': 'Sun, 22 Mar 2026 11:00:38 GMT', 'content-type': 'application/json', 'content-length': '20', 'connection': 'keep-alive', 'x-pinecone-request-lsn': '1', 'x-pinecone-request-logical-size': '111608', 'x-pinecone-request-latency-ms': '1355', 'x-envoy-upstream-service-time': '504', 'x-pinecone-response-duration-ms': '1383', 'grpc-status': '0', 'server': 'envoy'}})